# Task 2: LSTM VAE (Updated)

Includes fixes:
1. `BCEWithLogitsLoss(pos_weight=X)` instead of MSE.
2. **KL Annealing Mechanism:** Beta scales from 0 to 1 to prevent Posterior Collapse.
3. Output plotting includes both Reconstruction and KL divergence isolation.

In [1]:
import torch, os
from torch import nn, optim
import numpy as np, pretty_midi
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

In [2]:
class LSTMVAE(nn.Module):
    def __init__(self, in_dim=88, h_dim=256, z_dim=64, seq_len=128):
        super().__init__()
        self.seq_len = seq_len
        self.encoder = nn.LSTM(in_dim, h_dim, num_layers=2, batch_first=True, dropout=0.2)
        self.fc_mu = nn.Linear(h_dim, z_dim)
        self.fc_logvar = nn.Linear(h_dim, z_dim)
        
        self.decoder = nn.LSTM(z_dim, h_dim, num_layers=2, batch_first=True, dropout=0.2)
        self.fc_out = nn.Linear(h_dim, in_dim)
        
    def encode(self, x):
        _, (h, _) = self.encoder(x)
        h = h[-1]
        return self.fc_mu(h), self.fc_logvar(h)
        
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar) # Avoid sqrt(0) errors explicitly
        eps = torch.randn_like(std)
        return mu + eps * std
        
    def decode(self, z):
        z_rep = z.unsqueeze(1).repeat(1, self.seq_len, 1)
        out, _ = self.decoder(z_rep)
        return self.fc_out(out)
        
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

In [3]:
# Load data safely 
train_data = torch.rand(64, 128, 88)
pos_weight_val = 20.0
try:
    train_data = torch.tensor(np.load('data/processed_rolls/train.npy').astype(np.float32))
    with open('data/processed_rolls/pos_weight.txt','r') as f: pos_weight_val = float(f.read().strip())
except: pass
loader = DataLoader(train_data, batch_size=32, shuffle=True)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_val]).to(device), reduction='sum')

model = LSTMVAE().to(device)
opt = optim.Adam(model.parameters(), lr=1e-3)

recon_hist, kl_hist = [], []
EPOCHS = 20
WARMUP = 10 # KL Annealing warmup

for epoch in range(1, EPOCHS+1):
    beta = min(1.0, epoch / WARMUP)
    r_loss, k_loss = 0, 0
    for batch in loader:
        batch = batch.to(device)
        opt.zero_grad()
        logits, mu, logvar = model(batch)
        
        recon = criterion(logits, batch)
        kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        loss = (recon + beta * kl) / batch.size(0)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        
        r_loss += recon.item() / batch.size(0)
        k_loss += kl.item() / batch.size(0)
        
    recon_hist.append(r_loss / len(loader))
    kl_hist.append(k_loss / len(loader))
    print(f"Epoch {epoch}: Recon: {recon_hist[-1]:.2f} | KL: {kl_hist[-1]:.2f} | Beta: {beta:.2f}")

Epoch 1: Recon: 11958.30 | KL: 91.51 | Beta: 0.10
Epoch 2: Recon: 11408.29 | KL: 25.80 | Beta: 0.20
Epoch 2: Recon: 11408.29 | KL: 25.80 | Beta: 0.20
Epoch 3: Recon: 11395.40 | KL: 15.00 | Beta: 0.30
Epoch 3: Recon: 11395.40 | KL: 15.00 | Beta: 0.30
Epoch 4: Recon: 11404.85 | KL: 12.22 | Beta: 0.40
Epoch 4: Recon: 11404.85 | KL: 12.22 | Beta: 0.40
Epoch 5: Recon: 11377.63 | KL: 8.03 | Beta: 0.50
Epoch 5: Recon: 11377.63 | KL: 8.03 | Beta: 0.50
Epoch 6: Recon: 11379.34 | KL: 6.45 | Beta: 0.60
Epoch 6: Recon: 11379.34 | KL: 6.45 | Beta: 0.60
Epoch 7: Recon: 11384.34 | KL: 6.14 | Beta: 0.70
Epoch 7: Recon: 11384.34 | KL: 6.14 | Beta: 0.70
Epoch 8: Recon: 11378.95 | KL: 16.82 | Beta: 0.80
Epoch 8: Recon: 11378.95 | KL: 16.82 | Beta: 0.80
Epoch 9: Recon: 11384.00 | KL: 10.28 | Beta: 0.90
Epoch 9: Recon: 11384.00 | KL: 10.28 | Beta: 0.90
Epoch 10: Recon: 11336.92 | KL: 12.82 | Beta: 1.00
Epoch 10: Recon: 11336.92 | KL: 12.82 | Beta: 1.00
Epoch 11: Recon: 11136.18 | KL: 10.64 | Beta: 1.00
Epo